# BirdCLEF+ 2025 - Pretrained models: noisy-student pseudo-labels on ALL train_soundscapes

**CPU, internet off, no training.** This notebook combines teacher outputs that were already computed on the GPU into soft pseudo-labels for a student.

**Inputs:** `ns-perch-all` (Perch 2.0 embeddings of every window), `ns-a1-all` (original A1 sigmoid probabilities of every window), the ORIGINAL A2 head `perch_head.npz` (`a2-perch-embed`) and the A1-distilled A2 head `perch_head_a1distill.npz` (`a2-a1distill`).

**Teachers:** `p_A1` = `a1_all_probs.npy`; `p_A2` = original A2 head on the Perch embeddings; `p_A2d` = distilled head on the same embeddings. The heads use their own saved mean/std and the shared numpy `head_predict`.

**MAIN** = mean(`p_A1`, `p_A2`). **ABLATION** = `p_A2d`. For both: `q = p ** 1.3`, `q[q < 0.01] = 0`. There is no temporal smoothing.

**Checks:** the two `soundscape_all_windows.csv` files are byte-identical and match both arrays (116712 rows expected), and the species lists of A1, both A2 heads and `sample_submission.csv` are identical.

**Output:** `pseudo_main.npy`, `pseudo_ablation.npy` (float32, N x 206), `soundscape_all_windows.csv`, `species.json`, `pseudo_labels_meta.json` (written last).

In [ ]:
import os, time
T_NOTEBOOK = time.time()
import gc, glob, hashlib, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata

SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
# On Kaggle every check is on. M3_STRICT only re-enables named checks inside a local smoke test.
STRICT = set(filter(None, os.environ.get("M3_STRICT", "").split(",")))
def strict(name):
    return not SMOKE or name in STRICT

OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
EXPECTED_WINDOWS = 116712
EXPECTED_CLASSES = 206
PERCH_DIM = 1536
POWER, FLOOR = 1.3, 0.01
CHUNK = int(os.environ.get("M3_CHUNK", "8192"))        # rows per head forward pass; bounds the memory used
SPEARMAN_N, SEED = 20000, 42
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 30 * 60))
SAFETY = 1.5

KERNELS = dict(perch_all="sc4000-birdclef-m3-ns-perch-all", a1_all="sc4000-birdclef-m3-ns-a1-all",
               a2="sc4000-birdclef-m3-a2-perch-embed", a2d="sc4000-birdclef-m3-a2-a1distill")

def find_in_kernel(name, kernel, env):
    """Folder of `name` inside the output of one attached kernel (several kernels write species.json)."""
    if os.environ.get(env):       # local tests only
        d = Path(os.environ[env])
        assert (d / name).exists(), f"{name} not in {d}"
        return d
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise FileNotFoundError(f"expected one {name} under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if len(hits) != 1:
        raise FileNotFoundError(f"expected exactly one {marker} under /kaggle/input, found {hits}")
    return Path(hits[0]).parent

def check_budget(projected_s, where):
    msg = f"{where}: projected notebook time {projected_s:.0f}s = {projected_s / 60:.1f} min (limit {BUDGET_S / 60:.0f} min)"
    print("TIMING GUARD |", msg, flush=True)
    if projected_s > BUDGET_S:
        raise RuntimeError(f"Projected time is over the limit. {msg}")

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Keep identical to the inference notebooks."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
PERCH_DIR = find_in_kernel("ns_perch_meta.json", KERNELS["perch_all"], "M3_PERCH_ALL_DIR")
A1_DIR = find_in_kernel("ns_a1_meta.json", KERNELS["a1_all"], "M3_A1_ALL_DIR")
A2_DIR = find_in_kernel("perch_head.npz", KERNELS["a2"], "M3_A2_DIR")
A2D_DIR = find_in_kernel("perch_head_a1distill.npz", KERNELS["a2d"], "M3_A2D_DIR")
print("Data:", DATA, "\nPerch all:", PERCH_DIR, "\nA1 all:", A1_DIR, "\nA2 head:", A2_DIR, "\nA2d head:", A2D_DIR, "\nCPUs:", os.cpu_count(), "| smoke:", SMOKE)

## Consistency checks (rows and species) before any computation

In [ ]:
t0 = time.time()
csv_perch, csv_a1 = PERCH_DIR / "soundscape_all_windows.csv", A1_DIR / "soundscape_all_windows.csv"
if csv_perch.read_bytes() != csv_a1.read_bytes():
    raise RuntimeError("the two soundscape_all_windows.csv files are not byte-identical")
CSV_SHA = sha256_of(csv_a1)
windows = pd.read_csv(csv_a1)
assert list(windows.columns) == ["filename", "win"], list(windows.columns)
N = len(windows)
if strict("n") and N != EXPECTED_WINDOWS:
    raise RuntimeError(f"expected {EXPECTED_WINDOWS} windows, the csv has {N}")
assert not windows.duplicated().any(), "duplicate (filename, win) rows"

meta_perch = json.load(open(PERCH_DIR / "ns_perch_meta.json"))
meta_a1 = json.load(open(A1_DIR / "ns_a1_meta.json"))
assert meta_perch["windows"] == N and meta_a1["windows"] == N, (meta_perch["windows"], meta_a1["windows"], N)
fail_p = sorted(f["filename"] for f in meta_perch["failed_files"]); fail_a = sorted(f["filename"] for f in meta_a1["failed_files"])
assert fail_p == fail_a, f"decode failure lists differ: perch {fail_p} vs A1 {fail_a}"

E = np.load(PERCH_DIR / "perch_all_embeddings.npy", mmap_mode="r")      # about 0.7 GB, read in chunks
assert E.shape == (N, PERCH_DIM) and E.dtype == np.float32, (E.shape, E.dtype)
p_A1 = np.load(A1_DIR / "a1_all_probs.npy")
assert p_A1.dtype == np.float32 and p_A1.ndim == 2 and len(p_A1) == N, (p_A1.dtype, p_A1.shape)
if not (np.isfinite(p_A1).all() and p_A1.min() >= 0 and p_A1.max() <= 1):
    raise RuntimeError("a1_all_probs.npy is non-finite or outside [0, 1]")

species = json.load(open(A1_DIR / "species.json"))
sp_a2 = json.load(open(A2_DIR / "species.json"))
sp_a2d = json.load(open(A2D_DIR / "species.json"))
sp_sub = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
assert species == sp_a2, "A2 (perch-embed) species list differs from A1's"
assert species == sp_a2d, "A2 a1distill species list differs from A1's"
assert species == sp_sub, "A1 species list differs from sample_submission"
C = len(species)
if strict("classes") and C != EXPECTED_CLASSES:
    raise RuntimeError(f"expected {EXPECTED_CLASSES} classes, found {C}")
assert p_A1.shape == (N, C), p_A1.shape

def load_head(path):
    head = dict(np.load(path))
    n_layers = int(head["n_layers"])
    assert head["mean"].shape == (PERCH_DIM,) and head["std"].shape == (PERCH_DIM,), (head["mean"].shape, head["std"].shape)
    assert head[f"W{n_layers - 1}"].shape[0] == C, f"{path.name}: last layer has {head[f'W{n_layers - 1}'].shape[0]} outputs, expected {C}"
    assert all(np.isfinite(v).all() for v in head.values()), f"{path.name}: non-finite arrays"
    return head, dict(path=str(path), sha256=sha256_of(path), n_layers=n_layers,
                      shapes={k: list(np.shape(v)) for k, v in head.items()})

head_a2, info_a2 = load_head(A2_DIR / "perch_head.npz")
head_a2d, info_a2d = load_head(A2D_DIR / "perch_head_a1distill.npz")
CHECK_S = time.time() - t0
print(f"checks passed: {N} windows, {windows['filename'].nunique()} files, {C} classes | csv sha256 {CSV_SHA[:12]} | "
      f"A2 head layers {info_a2['n_layers']}, A2d head layers {info_a2d['n_layers']} | {CHECK_S:.1f}s", flush=True)

## Head forward passes (numpy, chunked)

In [ ]:
n_chunks = int(np.ceil(N / CHUNK))

def run_head(head, label, heads_left_after):
    t = time.time()
    out = np.empty((N, C), np.float32)
    for k, s in enumerate(range(0, N, CHUNK)):
        x = np.asarray(E[s:s + CHUNK], dtype=np.float32)
        if not np.isfinite(x).all():
            raise RuntimeError(f"non-finite embeddings in rows {s}..{s + len(x)}")
        with np.errstate(over="ignore"):      # exp(+large) -> inf -> probability 0, which is correct
            out[s:s + len(x)] = head_predict(head, x)
        if k == 0:
            per_chunk = time.time() - t
            remaining = per_chunk * ((n_chunks - 1) + heads_left_after * n_chunks)
            check_budget(time.time() - T_NOTEBOOK + SAFETY * remaining + 60, f"{label}, after chunk 1 of {n_chunks}")
    if not (np.isfinite(out).all() and out.min() >= 0 and out.max() <= 1):
        raise RuntimeError(f"{label}: probabilities non-finite or outside [0, 1]")
    secs = time.time() - t
    print(f"{label}: {N} windows in {secs:.1f}s", flush=True)
    return out, secs

p_A2, A2_S = run_head(head_a2, "A2 (original head)", 1)
p_A2d, A2D_S = run_head(head_a2d, "A2d (a1distill head)", 0)
del E
gc.collect()

## Pseudo-labels and statistics

In [ ]:
def sharpen(p):
    q = np.power(p, POWER).astype(np.float32)
    q[q < FLOOR] = 0
    return q

def label_stats(q):
    n_nonzero = (q > 0).sum(axis=0); n_half = (q >= 0.5).sum(axis=0); q_sum = q.sum(axis=0, dtype=np.float64)
    order = np.lexsort((-q_sum, -n_half))[:10]        # most windows with a label >= 0.5, ties by summed label
    return dict(pct_nonzero=float(100 * (q > 0).mean()), windows_any_label_ge_0_5=int((q.max(axis=1) >= 0.5).sum()),
                mean_prob=float(q.mean(dtype=np.float64)), mean_max_per_window=float(q.max(axis=1).mean(dtype=np.float64)),
                top10_by_windows_ge_0_5=[dict(species=species[i], windows_ge_0_5=int(n_half[i]), windows_nonzero=int(n_nonzero[i]),
                                              sum_label=float(q_sum[i])) for i in order])

t0 = time.time()
raw_means = dict(p_A1=float(p_A1.mean(dtype=np.float64)), p_A2=float(p_A2.mean(dtype=np.float64)), p_A2d=float(p_A2d.mean(dtype=np.float64)))
print("raw (pre-sharpening) mean probability:", {k: round(v, 5) for k, v in raw_means.items()}, flush=True)

# Informational: per-class Spearman between p_A1 and p_A2 on a random sample of windows.
rng = np.random.default_rng(SEED)
idx = np.sort(rng.choice(N, size=min(SPEARMAN_N, N), replace=False))
ra, rb = rankdata(p_A1[idx], axis=0), rankdata(p_A2[idx], axis=0)
ra -= ra.mean(axis=0); rb -= rb.mean(axis=0)
den = np.sqrt((ra ** 2).sum(axis=0) * (rb ** 2).sum(axis=0))
with np.errstate(invalid="ignore", divide="ignore"):
    rho = np.where(den > 0, (ra * rb).sum(axis=0) / den, np.nan)      # constant column -> undefined
valid = np.isfinite(rho)
spearman = dict(n_windows=int(len(idx)), seed=SEED, n_classes_defined=int(valid.sum()),
                mean=float(np.nanmean(rho)) if valid.any() else None, median=float(np.nanmedian(rho)) if valid.any() else None,
                min=float(np.nanmin(rho)) if valid.any() else None, max=float(np.nanmax(rho)) if valid.any() else None,
                per_class={species[i]: (float(rho[i]) if valid[i] else None) for i in range(C)})
print(f"Spearman p_A1 vs p_A2 per class ({len(idx)} windows): mean {spearman['mean']} | median {spearman['median']} | "
      f"defined for {spearman['n_classes_defined']}/{C} classes", flush=True)
del ra, rb

main = sharpen((p_A1 + p_A2) / 2)
ablation = sharpen(p_A2d)
for name, q in (("main", main), ("ablation", ablation)):
    assert q.dtype == np.float32 and q.shape == (N, C) and np.isfinite(q).all() and q.min() >= 0 and q.max() <= 1, name

stats = dict(main=label_stats(main), ablation=label_stats(ablation),
             # per teacher, same sharpening and floor, for comparison
             p_A1=label_stats(sharpen(p_A1)), p_A2=label_stats(sharpen(p_A2)))
for name, s in stats.items():
    print(f"[{name}] non-zero {s['pct_nonzero']:.3f}% | windows with any label >= 0.5: {s['windows_any_label_ge_0_5']} | "
          f"mean prob {s['mean_prob']:.5f} | mean per-window max {s['mean_max_per_window']:.4f}", flush=True)
    print("   top10:", [(r["species"], r["windows_ge_0_5"]) for r in s["top10_by_windows_ge_0_5"]], flush=True)
STATS_S = time.time() - t0

## Save (temporary names first, meta file last)

In [ ]:
def publish(tmp, final):
    os.replace(tmp, OUT / final)

np.save(OUT / "pseudo_main.tmp.npy", main)
np.save(OUT / "pseudo_ablation.tmp.npy", ablation)
shutil.copyfile(csv_a1, OUT / "soundscape_all_windows.tmp.csv")
with open(OUT / "species.tmp.json", "w") as f:
    json.dump(species, f)
for a, b in (("pseudo_main.tmp.npy", main), ("pseudo_ablation.tmp.npy", ablation)):
    back = np.load(OUT / a)
    assert back.dtype == np.float32 and np.array_equal(back, b), a
assert sha256_of(OUT / "soundscape_all_windows.tmp.csv") == CSV_SHA
publish(OUT / "pseudo_main.tmp.npy", "pseudo_main.npy")
publish(OUT / "pseudo_ablation.tmp.npy", "pseudo_ablation.npy")
publish(OUT / "soundscape_all_windows.tmp.csv", "soundscape_all_windows.csv")
publish(OUT / "species.tmp.json", "species.json")

total = time.time() - T_NOTEBOOK
meta = dict(
    rows="row i of both arrays = row i of soundscape_all_windows.csv (filename, win); 5 s windows, sorted files",
    n_windows=N, n_classes=C, windows_csv_sha256=CSV_SHA, failed_files=fail_a,
    sources=dict(kernels=KERNELS, a2_head=info_a2, a2d_head=info_a2d),
    main="sharpen(mean(p_A1, p_A2))", ablation="sharpen(p_A2d)",
    sharpen=dict(power=POWER, floor=FLOOR, rule="q = p ** power; q[q < floor] = 0"), temporal_smoothing=False,
    p_A1="ns-a1-all a1_all_probs.npy (original A1, raw sigmoid)",
    p_A2="perch_head.npz (a2-perch-embed, original A2) on perch_all_embeddings.npy, numpy head_predict",
    p_A2d="perch_head_a1distill.npz (a2-a1distill) on perch_all_embeddings.npy, numpy head_predict",
    raw_mean_prob=raw_means, spearman_p_A1_vs_p_A2=spearman, stats_after_sharpening=stats,
    seconds=dict(checks=CHECK_S, head_a2=A2_S, head_a2d=A2D_S, stats=STATS_S, notebook=total))
with open(OUT / "pseudo_labels_meta.tmp.json", "w") as f:
    json.dump(meta, f, indent=1)
publish(OUT / "pseudo_labels_meta.tmp.json", "pseudo_labels_meta.json")     # written last: the run is complete
print(f"TIMING | {N} windows, {C} classes | checks {CHECK_S:.0f}s | A2 head {A2_S:.0f}s | A2d head {A2D_S:.0f}s | "
      f"stats {STATS_S:.0f}s | notebook {total:.0f}s = {total / 60:.1f} min | CPU")